In [12]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

panels = {"../data/f3a.csv": "Top 10%", "../data/f3b.csv": "Top 1%", "../data/f3c.csv": "Top 0.1%"}
frames = []
for path, label in panels.items():
    d = pd.read_csv(path); d.columns = ["decade", "share", "source"]; d["panel"] = label
    frames.append(d)
df = pd.concat(frames, ignore_index=True)
df["decade"] = pd.to_datetime(df["decade"], format="%Y")

panel_order  = ["Top 10%", "Top 1%", "Top 0.1%"]
source_order = ["Observed", "True"]
colors = ["#e45756", "#3aa8a0"]     # Observed red, True teal
shapes = ["circle", "triangle"]

styles = EcoStyles()
styles.register_and_enable_theme()

legend = alt.Legend(title="Source", orient="bottom")

base = alt.Chart().encode(
    x=alt.X("decade:T", title="Decade", axis=alt.Axis(format="%Y", tickCount=6)),
    y=alt.Y("share:Q", title="Share"),
    color=alt.Color("source:N", sort=source_order,
                    scale=alt.Scale(domain=source_order, range=colors), legend=legend),
)
line = base.mark_line(strokeWidth=1.6)
pts  = base.mark_point(filled=True, size=55).encode(
    shape=alt.Shape("source:N", sort=source_order,
                    scale=alt.Scale(domain=source_order, range=shapes), legend=legend))

chart = (
    alt.layer(line, pts, data=df)
    .properties(width=560, height=115)
    .facet(alt.Facet("panel:N", title=None, sort=panel_order,
                     header=alt.Header(labelFontWeight="bold", labelFontSize=13, labelAnchor="start")),
           columns=1)
    .resolve_scale(y="independent")     # x stays shared -> one axis at the bottom
)
styles.save(chart, name="uk_top_shares_observed_true", svg=True)
chart

alt.FacetChart(...)